In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/embeddings-lab/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Exercises 03 · Geometry diagnostics
Implement three diagnostics that you can run on any embedding matrix at work.
The solutions are in `../solutions/ex03.ipynb`.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

## Task 1 — participation ratio (effective dimensionality)
Calculate $\mathrm{PR} = \left(\sum_i \lambda_i\right)^2 / \sum_i \lambda_i^2$ over the eigenvalues of the covariance of `X`.
Isotropic $d$-dim data gives $\approx d$. Rank-1 data gives $\approx 1$.

In [2]:
def participation_ratio(X):
    Xc = X - X.mean(0)
    lam = np.linalg.svd(Xc, compute_uv=False) ** 2
    return float(lam.sum() ** 2 / (lam ** 2).sum())

iso = rng.normal(size=(3000, 10))
r1 = np.outer(rng.normal(size=3000), rng.normal(size=10)) + 1e-3 * rng.normal(size=(3000, 10))
assert 8.5 < participation_ratio(iso) < 10.5
assert participation_ratio(r1) < 1.5
print(f"participation_ratio ✓  (isotropic≈{participation_ratio(iso):.1f}, rank-1≈{participation_ratio(r1):.2f})")

participation_ratio ✓  (isotropic≈10.0, rank-1≈1.00)


## Task 2 — CSLS rescoring (the hubness fix)
$\operatorname{csls}(S, k) = 2 \cdot S - r(\text{row}) - r(\text{col})$, where ${r(x)}$ is the mean of the top-$k$
similarities of $x$ (diagonal excluded). CSLS gives a lower score to points that are
near *everything*.

In [3]:
def csls(S, k=10):
    S2 = S.copy()
    np.fill_diagonal(S2, -np.inf)
    r = np.sort(S2, axis=1)[:, -k:].mean(1)
    return 2 * S2 - r[None, :] - r[:, None]

St = np.array([[1.0, 0.9, 0.1], [0.9, 1.0, 0.2], [0.1, 0.2, 1.0]])
out = csls(St, k=1)          # r = [0.9, 0.9, 0.2]
assert np.isclose(out[0, 1], 2 * 0.9 - 0.9 - 0.9)
assert np.isclose(out[0, 2], 2 * 0.1 - 0.9 - 0.2)
print("csls ✓")

csls ✓


## Task 3 — All-but-the-Top (Mu & Viswanath 2018)
Center `X`. Then remove its projection onto the top `n_pc` principal components.
The function returns the corrected matrix.

In [4]:
def all_but_the_top(X, n_pc=2):
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc - (Xc @ Vt[:n_pc].T) @ Vt[:n_pc]

X = rng.normal(size=(500, 20)) + 5 * np.outer(rng.normal(size=500), np.ones(20))
Y = all_but_the_top(X, 1)
assert np.allclose(Y.mean(0), 0, atol=1e-8)
top_dir = np.linalg.svd(X - X.mean(0), full_matrices=False)[2][0]
assert np.abs(Y @ top_dir).max() < 1e-6      # no variance left along old top PC
print("all_but_the_top ✓")

all_but_the_top ✓


## Task 4 (open) — apply them
Load `../artifacts/word_vectors.npz` and report these items:

- the PR of the SGNS vectors,
- the five largest hubs before and after CSLS,
- if all-but-the-top changes the related-vs-unrelated AUC from notebook 03.

Write two or three sentences about the results.